In [6]:
import os
import sqlite3
import pandas as pd
from tqdm import tqdm
from langchain_community.document_loaders import CSVLoader #
from langchain_core.documents import Document #
from langchain_text_splitters import RecursiveCharacterTextSplitter #
from langchain_ollama import OllamaEmbeddings #
from langchain_chroma import Chroma #


# VECTORIZE

## ADDRESS PREP

In [21]:
location_mukim_district_state = pd.read_csv('../output/malaysia-postcodes-location-mukim-district-state.csv',dtype='string')
location_mukim_district_state['address'] = location_mukim_district_state.apply(
    lambda row: f"{row['location']}, {row['mukim']}, {row['postcode']}, {row['district']}, {row['state']}", axis=1
) 

location_mukim_district_state['address'] = location_mukim_district_state['address'].apply(lambda x: x.split(', '))
location_mukim_district_state['address'] = location_mukim_district_state['address'].apply(lambda x: [i for i in x if i != '<NA>'])
location_mukim_district_state['address'] = location_mukim_district_state['address'].apply(lambda x: list(dict.fromkeys(x)))
location_mukim_district_state['address'] = location_mukim_district_state['address'].apply(lambda x: ', '.join(x))
address=location_mukim_district_state[['address']]
address.to_csv('../output/address.csv', index=False)

## ADDRESS EMBEDDING

In [22]:
file_path_csv = '../output/address.csv'

# Create a CSVLoader instance
loader = CSVLoader(file_path=file_path_csv)
loader

### CREATE DOCUMENT

In [23]:
# Load documents from CSV file
documents = loader.load()
documents

[Document(metadata={'source': '../output/address.csv', 'row': 0}, page_content='address: ABI, 01000, KANGAR, PERLIS'),
 Document(metadata={'source': '../output/address.csv', 'row': 1}, page_content='address: ARAU, 02600, PERLIS'),
 Document(metadata={'source': '../output/address.csv', 'row': 2}, page_content='address: BERSERI, 02400, KANGAR, PERLIS'),
 Document(metadata={'source': '../output/address.csv', 'row': 3}, page_content='address: CHUPING, 02500, PADANG BESAR, PERLIS'),
 Document(metadata={'source': '../output/address.csv', 'row': 4}, page_content='address: UTAN AJI, 01000, KANGAR, PERLIS'),
 Document(metadata={'source': '../output/address.csv', 'row': 5}, page_content='address: JEJAWI, 01000, ARAU, PERLIS'),
 Document(metadata={'source': '../output/address.csv', 'row': 6}, page_content='address: KAYANG, 02000, KANGAR, PERLIS'),
 Document(metadata={'source': '../output/address.csv', 'row': 7}, page_content='address: KECHOR, 01000, KANGAR, PERLIS'),
 Document(metadata={'source':

In [24]:
documents[0]

Document(metadata={'source': '../output/address.csv', 'row': 0}, page_content='address: ABI, 01000, KANGAR, PERLIS')

In [25]:
documents[0].page_content[:1000]  # Display the first 1000 characters of the first document


'address: ABI, 01000, KANGAR, PERLIS'

In [26]:
print(len(documents))
total_docs  = len(documents)

58394


In [31]:
# Initialize the embedding
oembed = OllamaEmbeddings(base_url="http://localhost:11434", model="llama3.2:latest")

In [28]:
# Define the folder path for Chroma's in-memory storage
persist_directory = "../output/vectorstore"

In [32]:

for i in tqdm(range(0, len(documents))):
    vectorstore = Chroma.from_documents(
        documents=[documents[i]], 
        embedding=oembed, 
        persist_directory=persist_directory,
        collection_name="base_address"  # Specify the collection name here
    )
print('Data Ingested into Vectorstore')


100%|██████████| 58394/58394 [33:15:19<00:00,  2.05s/it]   

Data Ingested into Vectorstore
